In [ ]:
# bPe_BCF_zevo

# Does the BFC model reproduce FLAMINGO's <bPe>(z) when fed the
# FLAMINGO-fitted parameters?

# Separate from ProfileComparisonv2, which asks a different question (do David's and Michael's codes agree with each other). 
# Only Michael's code is used here - it exposes all three non-thermal parameters in par.baryon, where David's has no b_nth.

# 1) Imports 

import sys, os
sys.path.append('..') # repo root: bfc_* modules
sys.path.append('../david_sandbox') # David's utils

import numpy as np
import matplotlib.pyplot as plt
import pyccl as ccl
from scipy.integrate import simpson

import bfc_pressure_profile_quicker as bcf_quick   # Michael's
import bfc_functions as bfc_f
from flamingo import Flamingo                       # local copy


# 2) - FLAMINGO curves 

FLAM_DATA = os.environ.get('FLAM_DATA', '/path/to/FLAMINGO')

flam_curves = {}
for v in ["fgas+2sigma", # low feedback
        "L1_m9", # standard feedback (fiducial)
        "fgas-2sigma", # stronger feedback
        "fgas-4sigma",
        "fgas-8sigma", # strongest feedback
        "LS8", # low-S8 cosmology, standard feedback
        "LS8_fgas-8sigma" # strongest feedback
          ]: 
    z_f, b_f, bg_f, _ = Flamingo(FLAM_DATA).bPe_of_z_full(v, every=1, nmax=1, template="nonlin")
    flam_curves[v] = (z_f, b_f)
    print(f"{v}: {len(z_f)} snapshots, z up to {z_f.max():.2f}")



# 3) - cosmology and shared halo-model ingredients

# Each FLAMINGO run has its own cosmology, so the BFC model has to use the same one as the run it's compared against. 


COSMO_PARAMS = {
        "D3A": dict(h=0.681, Omega_b=0.0486, Omega_c=0.306-0.0486,
                    A_s=2.099e-9, n_s=0.967, m_nu=0.06),
        "LS8": dict(h=0.682, Omega_b=0.0473, Omega_c=0.305-0.0473,
                    A_s=1.836e-9, n_s=0.965, m_nu=0.06)}

_cosmo_cache = {}

def get_cosmo(sim):
    """
    CCL cosmology for a BFC parameter set. m9 and its feedback variants ran in
    D3A; anything LS8-flavoured ran in LS8.

    • ** unpacks the dict into keyword args: ccl.Cosmology(h=0.681, ...)
    • cached because building a ccl.Cosmology is expensive and bPe() rebuilds
      a profile at every redshift.
    """
    key = "LS8" if sim.startswith("LS8") else "D3A"
    if key not in _cosmo_cache:
        _cosmo_cache[key] = ccl.Cosmology(**COSMO_PARAMS[key])
    return _cosmo_cache[key]




cM  = ut.ConcentrationDutton14(mass_def='200c')
bM  = ccl.halos.HaloBiasTinker10(mass_def='200c')
nM  = ccl.halos.MassFuncBocquet20(mass_def='200c')
hmc = ccl.halos.HMCalculator(mass_function=nM, halo_bias=bM, mass_def='200c')





# 4) - bPe machinery (Michael's class has no volint, so supply it)

def volint_generic(prof, cosmo, xmax, M, a, xmin=1e-3, nx=100):
    """
    Volume integral Int[4 pi r^2 rho(r) dr], as a port of David's volint.
    Written in d ln x, hence xs**3 rather than xs**2 (the extra power is
    the Jacobian).
    """

    M_use = np.atleast_1d(M)
    rvir  = prof.mass_def.get_radius(cosmo, M_use, a) / a
    xs  = np.geomspace(xmin, xmax, nx)
    lxs = np.log(xs)
    Vint = [simpson(np.squeeze(np.atleast_2d(prof._real(cosmo, xs*rv, Mm, a)))
                    * xs**3, x=lxs) * 4*np.pi*rv**3
            for Mm, rv in zip(M_use, rvir)]
    Vint = np.array(Vint)
    return np.squeeze(Vint, axis=0) if np.ndim(M) == 0 else Vint


def bU_generic(hmc, prof, z, cosmo, xmax=1000, xmin=1e-3, nx=100):
    """
    Mass-function- and bias-weighted integral of the volume integral.
    """

    a = 1/(1+z)
    hmc._get_ingredients(cosmo, a, get_bf=True)
    Ms = 10**hmc._lmass
    Uvol = volint_generic(prof, cosmo, xmax, Ms, a, xmin=xmin, nx=nx)
    return hmc._integrate_over_mbf(Uvol) / prof.get_normalization(cosmo, a, hmc=hmc)



# 5) - FLAMINGO-matched BFC parameters 

# The parameters now evolve with redshift.
FLAM_TABLE = {
    # z,      logMc,   mu,     delta,  theta_co, eta,     deta,   Nstar,  cIGA
    "m9": [ # fiducial, corresponds to L1_m9
        (0.0,  13.268, 0.755,  4.961,  0.262,    0.066,   0.215,  0.026,  0.122),
        (0.5,  13.419, 0.483,  6.053,  0.220,    0.061,   0.226,  0.025,  0.233),
        (1.0,  13.294, 0.343,  6.597,  0.259,    0.070,   0.234,  0.026,  0.382),
        (1.5,  12.704, 0.231,  6.106,  0.268,    0.053,   0.214,  0.023,  0.543),
    ],
    "m9_fgas-8s": [ # no non thermal pressure - as david quotes.
        (0.0,  13.517, 0.735,  4.220,  0.321,    0.073,   0.226,  0.028,  0.107),
        (0.5,  13.459, 0.520,  4.717,  0.319,    0.060,   0.240,  0.027,  0.254),
        (1.0,  13.519, 0.405,  5.314,  0.302,    0.078,   0.243,  0.029,  0.393),
        (1.5,  13.446, 0.313,  6.151,  0.434,    0.072,   0.249,  0.028,  0.580),
    ],
}

_KEYS = ["logMc", "mu", "delta", "theta_co", "eta", "deta", "Nstar", "cIGA"]



# non thermal params.
NTH = {
    "m9":         dict(a_nth=0.134,  n_nth=0.35, b_nth=0.37),  # amplitude / slope / z-evolution
    "m9_fgas-8s": dict(a_nth=0.0001, n_nth=2.00, b_nth=0.50),
}


SCALE = 1.12 # apparent difference between curves - to be investigated later in the code.


def flam_pars(z, sim="m9"):
    """
    function that linearly interpolates the above table to any z.

    """

    tab = np.array(FLAM_TABLE[sim]) # create the table into a 2D numpy array (for operation) for a specfic feedback
    zs = tab[:, 0] # all rows of first column

    # tab [:, n] - slicing - all rows of column n, i+1 as z is col 0 in table but first param is 0 in keys.
    # as it read: for each param name, k, pull out column, iterp that column  against zs, and store under name in a dict.
    return {k: float(np.interp(z, zs, tab[:, i + 1])) for i, k in enumerate(_KEYS)}


def make_mike_flam(z, sim="m9", kind='n_electron', **nth):
    """
    Michael-style profile: Table 1 params at z, non-thermal sector from NTH.

    • any a_nth/n_nth/b_nth passed as kwargs override the fitted values,
      which is only for poking at them - the default is Michael's fit.
    """
    p = flam_pars(z, sim)
    n = {**NTH[sim], **nth}

    par = bcf_quick.default_par()

    
    cp = COSMO_PARAMS["LS8" if sim.startswith("LS8") else "D3A"]
    par.cosmo.Om = cp['Omega_c'] + cp['Omega_b']

    b = par.baryon
    b.Mc = 10**p["logMc"]
    b.mu = p["mu"]
    b.delta = p["delta"]
    b.thco = p["theta_co"]
    b.eta = p["eta"]
    b.deta = p["deta"]
    b.Nstar = p["Nstar"]
    b.ciga = p["cIGA"]
    b.a_nth, b.n_nth, b.b_nth = n['a_nth'], n['n_nth'], n['b_nth']
    b.alpha, b.gamma, b.Mstar = 1.0, 1.5, 2.5e11 # not in Table 1
    return bcf_quick.HaloProfilePressureBFC(
        mass_def='200c', bias_model=bM, param=par,
        units='eV', little_h=False, comoving=False,
        concentration=cM, use_nfw_mass=True, skip_ac=True, kind=kind)


def bPe(zs, sim="m9", **nth):
    """
    <bPe> [meV/cm^3] evaluated at each z, in the cosmology of the FLAMINGO run
    this parameter set was fitted to. Includes the SCALE normalisation.
    """
    cos = get_cosmo(sim)
    return SCALE * np.array(
        [1e3 * bU_generic(hmc, make_mike_flam(z, sim, **nth), z, cos)
         for z in zs])


# 6) - the summary figure: BFC vs FLAMINGO vs data

zs = np.linspace(0.01, 1.5, 15)

# pair each BFC parameter set with the FLAMINGO run it was fitted to
PAIRS = [("m9", "L1_m9", 'C0'), ("m9_fgas-8s", "fgas-8sigma", 'C1')]

fig, ax = plt.subplots(1, 2, figsize=(14, 5.5),
                       gridspec_kw={'width_ratios': [3, 2]})

# use fiducial
for sim, ref, c in PAIRS:
    ax[0].plot(zs, bPe(zs, sim), '-', color=c, lw=2, label=f'BFC, {sim}')

# for each flamingo feedback , return redshift and bPe from solve_bPe_full function (using nonlin).
for v, (z_f, b_f) in flam_curves.items():
    sel = z_f <= 1.5 # make a cut
    ax[0].plot(z_f[sel], b_f[sel], '--', lw=2, alpha=0.6,
               color=Flamingo.COLORS.get(v), label=f'FLAMINGO {v}')


          #BGS mean redshifts.  # BGS bPe.      # BGS errors.  (measured - kmax = 0.3 values (n=1 - up to k^2 term))
ax[0].errorbar([0.1618, 0.3185], [0.125, 0.215], yerr=[0.023, 0.030],
               fmt='o', color='navy', label='BGS (this work)')

# same as above but for LRGs
ax[0].errorbar([0.47, 0.63, 0.79, 0.93], [0.341, 0.348, 0.415, 0.420],
               yerr=[0.039, 0.041, 0.066, 0.058],
               fmt='o', color='crimson', label='LRG')
ax[0].set_xlabel('z'); ax[0].set_ylabel(r'$\langle bP_e\rangle$ [meV/cm$^3$]')
ax[0].legend(fontsize=7, ncol=2)
ax[0].set_title(f"Table params + Michael's non-thermal best fit params (x{SCALE})")

# does the model actually track the sim it was fitted to? ratio ~ 1 = yes,
# a drift with z = the model can't capture the evolution even when fitted.
ax[1].axhline(0, color='k', lw=0.8)
for sim, ref, c in PAIRS:
    z_f, b_f = flam_curves[ref]
    srt = np.argsort(z_f)
    y_ref = np.interp(zs, z_f[srt], b_f[srt])
    ax[1].plot(zs, bPe(zs, sim) / y_ref - 1, '-o', color=c, ms=4,
               label=f'{sim} / {ref}')
ax[1].set_xlabel('z'); ax[1].set_ylabel('BFC / FLAMINGO  -  1')
ax[1].legend(fontsize=8); ax[1].set_title('fractional residual')
plt.tight_layout()


# Each entry is BFC divided by FLAMINGO. 1.0 means the model reproduces the simulation exactly.
print(f"{'z':>6} " + " ".join(f"{s+'/'+r:>20}" for s, r, _ in PAIRS)) # just for readability of table: For each pair it builds a string like "m9" + "/" + "L1_m9" → m9/L1_m9
for z in [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]: # Table 1's own redshifts plus the midpoints between them
    row = [] # one row per redshift.
    for sim, ref, _ in PAIRS: 
        z_f, b_f = flam_curves[ref]
        srt = np.argsort(z_f)

        # Numerator: the BFC model at this redshift. np.array([z]) wraps the scalar because bPe expects a sequence -  [0] takes it back out.
        # Denominator: FLAMINGO interpolated to the same redshift, since its own grid is the snapshot redshifts rather than these round numbers.
        row.append(bPe(np.array([z]), sim)[0] / np.interp(z, z_f[srt], b_f[srt]))
    print(f"{z:6.2f} " + " ".join(f"{r:20.4f}" for r in row))



In [ ]:
z_f, b_fid, _ , _ = Flamingo(FLAM_DATA).bPe_of_z_full("L1_m9", every=1, template="nonlin")
z_l, b_ls8, _ , _ = Flamingo(FLAM_DATA).bPe_of_z_full("LS8",   every=1, template="nonlin")

lnr = np.log(0.7713/0.8207) # from prints above
eps = lambda z: 6.65 - 5.56*(1+z) + 1.937*(1+z)**2 # Salcido Table 4

print(f"{'z':>6} {'ratio':>8} {'eps_meas':>10} {'eps_paper':>10}")
for i in range(len(z_f)):
    if not (0.1 <= z_f[i] <= 1.0): continue
    R = b_ls8[i]/b_fid[i]
    print(f"{z_f[i]:6.2f} {R:8.3f} {np.log(R)/lnr:10.2f} {eps(z_f[i]):10.2f}")

In [ ]:
FLAM_TABLE["LS8_m9"] = FLAM_TABLE["m9"] # same Table 1 params
NTH["LS8_m9"] = NTH["m9"] # same non-thermal fit
                                          

z_bcf = np.linspace(0.1, 1.0, 10)
b_d3a = bPe(z_bcf, "m9")
b_ls8 = bPe(z_bcf, "LS8_m9")

lnr = np.log(0.7713/0.8207) # from prints above
eps = lambda z: 6.65 - 5.56*(1+z) + 1.937*(1+z)**2 # J. Salcido et al 2026. Table 4

print(f"{'z':>6} {'ratio':>8} {'eps_BCF':>10} {'eps_paper':>10}")
for i, z in enumerate(z_bcf):
    R = b_ls8[i]/b_d3a[i]
    print(f"{z:6.2f} {R:8.3f} {np.log(R)/lnr:10.2f} {eps(z):10.2f}")

# Part 2: Compare Y5R500

In [ ]:
import swiftsimio.objects as sso, pickle as pkl, numpy as np # sso contains the cosmo_array class we need to modify.


# fetches an attribute - returns the default if it doesnt exist, First time running - theres no "is patched" so returns False. - therefore not false is true - block runs. - after its true and doesnt run.
# exists because of recursion limit error we had - rerunning without it captured the patched function as "the original", so it called itself forever.
if not getattr(sso.cosmo_array.__setstate__, '_is_patched', False):

    # save a reference to the real method before replacing it.
    _orig = sso.cosmo_array.__setstate__


    
    def _patched(self, state):
        """ 
        • patched function to help with version clashes
         __setstate__ is what python calls during unpicklling to restore an objects internals.
        """
        
        print(len(state), [type(x) for x in state])
        extra = state[0] # state[0] is the typle of swiftsio - specific metadata
        if len(extra) == 3: # if 3 entries - we add a 4th
            state = ((*extra, True), *state[1:]) # True - the defaukt for the missing valid_transgorm field - rebuild the state and hand it to og method- now finds 4 values itr expects.
            print(len(state), [type(x) for x in state])
        return _orig(self, state)
    
    
    # Tag it so the guard works next time, then install it in place of the real method.
    # Every cosmo_array unpickled from now on goes through the patch.
    _patched._is_patched = True
    sso.cosmo_array.__setstate__ = _patched





# saras file is a pickle - a python format for saving objects to disk. 
# The arrays in her file arent numpy - theyre swiftsimio - cosmo_array objects - they acarry units and cosmology metadata alongside the numbers. 
# saras version stores 3 peices of metadata, my version expects 4 - unpacking error.

FLAM_ANALYSIS = os.environ.get('FLAM_ANALYSIS', '/path/to/flamingo_analysis')
p = f'{FLAM_ANALYSIS}/L1000N1800/HYDRO_FIDUCIAL/snap_0077/Info_galaxiesSOAP-HBT.pickle'
f = pkl.load(open(p, 'rb')) # 'rb' = 'read binary'
print(sorted(f.keys()))

In [ ]:

# see what we have.
for k in sorted(f.keys()):
    
    v = f[k] # v is the swiftsimio array with its metadata - just the above 6 titled columns.
    arr = np.asarray(v) # make numpy array so operations work. 
    
    # print 'units' attribute 
    print(f"{k:26} n={len(arr):8d}  units={getattr(v,'units','—')}  "
          f"min={np.nanmin(arr):.4g}  max={np.nanmax(arr):.4g}  " # nanmin/nanmax ignores any potential nan values otherise this could return nan.
          f"median={np.nanmedian(arr[arr>0]):.4g}") # 

In [ ]:
# run this once - so it saves, doenst give stupid kernel issues and runs sometimes.
import pickle as pkl, numpy as np
cat = pkl.load(open(p, 'rb'))
np.savez_compressed('soap_fid_77.npz',
                    m200c=np.asarray(cat['m200c_central']),
                    Y5R500=np.asarray(cat['ComptonY_5xR500c_central']),
                    Y200c=np.asarray(cat['ComptonY_200c_central']),
                    m5xR500=np.asarray(cat['m5xR500c_central']))

In [ ]:
# Y - M200c : FLAMINGO SOAP catalogue vs the BFC model

# imports
import numpy as np
import matplotlib.pyplot as plt
import pyccl as ccl

# 1) read the catalogue



# read halo catalogue - saved in my extra space.
d = np.load(f'{BPE_DATA}/soap_fid_77.npz')
M200_cat = d['m200c'] * 1e10 # Msun. 
Y_cat = d['Y200c'] # Mpc^2




# needed for rescaled plot to send to micheal.





#n = nM(cos, 10**hmc._lmass, a) 
#b = bM(cos, 10**hmc._lmass, a) 


# 2) units


# Y = (sigma_T / m_e c^2) * Int Pe dV
# so inverting gives the integrated pressure as a total energy. Our


# volint_generic returns Pe in eV/cm^3 times Mpc^3, so convert into that.

sigmaT = 6.65245873e-25 / (3.086e24)**2 # Mpc^2   (Sara's constants - sent through slack)
mec2 = 0.511e9 # eV
mpc_cm = 3.0857e24 # cm per Mpc

y_to_volint = mec2 / sigmaT / 1e3 / mpc_cm**3 # ev/cm^3/ * Mpc^5

P_cat = Y_cat * y_to_volint # eV/cm^3 Mpc^3  

print(f"\n{len(M200_cat):,} haloes")
print(f"M200c: {M200_cat.min():.3g} to {M200_cat.max():.3g} Msun")
print(f"Y: {Y_cat.max():.3g} Mpc^2 max, {np.sum(Y_cat > 0):,} non-zero")
print(f"IntPeDV: {P_cat[P_cat>0].min():.3g} to {P_cat.max():.3g} eV/cm^3 Mpc^3")



# 3) bin the catalogue


z_cat = 0 # snap 77 redshift (z=0) - only one locally (according to sara)

a = 1/(1+z_cat); cos = get_cosmo("m9")
lnM = np.log(10)*hmc._lmass # hmc._lmasss is CCls log10M grid 

m_min = 1e11

good = (M200_cat > m_min) & (Y_cat > 0) # keep the mask and the printed
                                        
print(f"\n{np.sum(good):,} haloes above {m_min:.0e} Msun with Y > 0")

m_edges = np.geomspace(m_min, M200_cat[good].max(), 25)
m_cent, P_med, P_lo, P_hi, n_in = [], [], [], [], []
for lo, hi in zip(m_edges[:-1], m_edges[1:]): # create a sip array of bins
    s = good & (M200_cat >= lo) & (M200_cat < hi) # combine mask cut with mask to dtermine which bin we are in
    if np.sum(s) < 10: # skip bins too sparse to trust
        continue



    m_cent.append(np.sqrt(lo*hi)) # central mass
    P_med.append(np.median(P_cat[s])) # median Pressure for that mass bin (median as few outliers would drag the mean.)
    P_lo.append(np.percentile(P_cat[s], 16)) # 1sigma percentriles for a skwed distribution - where mean and standard deviation would be misled.
    P_hi.append(np.percentile(P_cat[s], 84))
    n_in.append(np.sum(s)) # amount in that bin

# make into arrays for array opertions.
m_cent, P_med = np.array(m_cent), np.array(P_med)
P_lo, P_hi, n_in = np.array(P_lo), np.array(P_hi), np.array(n_in)



# 4) the BFC side

# The aperture is 5*R500c but the profile is built on 200c, so we need R500c from M200c. micheals bcf_functiosn r500 function solves for it.


def Y_bfc(M200, z, sim="m9", nx=100):
    """
    Int Pe dV within 5*R500c, in eV/cm^3 * Mpc^3, for haloes of mass M200c.
    Same units as P_cat above.
    """
   
    a   = 1/(1+z)
    cos = get_cosmo(sim)
    prof = make_mike_flam(z, sim) # call above function that returns pressure rpfoile with ideal aprams to match flamingo
    M = np.atleast_1d(M200)

    r200 = prof.mass_def.get_radius(cos, M, a) / a # CCLs get radius returns comoving so division by the scale factor gives physical (Mpc).
    conc = cM(cos, M, a)
    r500 = np.array([float(np.atleast_1d(bfc_f.r500_fct(r, c))[0]) # r500_fct solves NFW equatrion numeraically for radius enclosing 500c - it uses fsolve - thus the array wrapping to extract a scalar.
                     for r, c in zip(r200, conc)])

    # David's volint seems to be in units of rvir.
    # rvir = self.mass_def.get_radius(blah blah) -  Since mass_def is '200c', the rvir" is r200c - hence dividing by r200 here.
    xmax = 5.0 * r500 / r200 # in units of r200 - as volint_generic uses these units.

    out = []
    for m, xm in zip(M, xmax):

        # Same volume integral as before, just stopping at the aperture rather than 1000rvir.
        out.append(float(volint_generic(prof, cos, xm, m, a, xmin=1e-3, nx=nx))) # float handles previosu index errors from volint (returns scalar)
    return np.array(out)

def Y_bfc_200(M200, z, sim="m9", nx=100):
    """
    Int Pe dV within r200c, in eV/cm^3 * Mpc^3.
    Simpler than the 5R500 version - the aperture is the profile's own rvir,
    so xmax = 1 with no NFW conversion needed.
    """
    a   = 1/(1+z)
    cos = get_cosmo(sim)
    prof = make_mike_flam(z, sim)
    M = np.atleast_1d(M200)
    return np.array([float(volint_generic(prof, cos, 1.0, m, a, xmin=1e-3, nx=nx))
                     for m in M])

# define M grid
M_grid = np.geomspace(m_min, 3e15, 20)



# call function at redshift 0 (snapshot 77)(fiducial)
P_bfc = Y_bfc_200(M_grid, z_cat, "m9") # - BFC model only
P_bfc_8s = Y_bfc_200(M_grid, z_cat, "m9_fgas-8s") # same but for stronger fedback model.








# 5) The comparsion
fig, ax = plt.subplots(1, 2, figsize=(13, 5),
                       gridspec_kw={'width_ratios': [3, 2]})

# weight by the <bPe> integrand so the plot shows where the signal comes from.
# NB this leaves the RATIO panel unchanged - n*b cancels when you divide.
w_cat = nM(cos, m_cent, a) * bM(cos, m_cent, a)
w_mod = nM(cos, M_grid, a) * bM(cos, M_grid, a)

ax[0].fill_between(m_cent, P_lo*w_cat, P_hi*w_cat, color='k', alpha=0.15,
                   label='FLAMINGO 16-84%')
ax[0].plot(m_cent, P_med * w_cat, 'k-o', ms=4, label='FLAMINGO median')
ax[0].plot(M_grid, P_bfc * w_mod, 'C0-', lw=2, label='BFC m9')
ax[0].plot(M_grid, P_bfc_8s * w_mod, 'C1-', lw=2, label='BFC m9_fgas-8s')
ax[0].set_ylabel(r'$\frac{dn}{dM}\,b\,\int P_e dV$  [eV cm$^{-3}$ M$_\odot^{-1}$]')

"""

ax[0].fill_between(m_cent, P_lo, P_hi, color='k', alpha=0.15,
                   label='FLAMINGO 16-84%')
ax[0].plot(m_cent, P_med, 'k-o', ms=4, label='FLAMINGO median')
ax[0].plot(M_grid, P_bfc,    'C0-', lw=2, label='BFC m9')
ax[0].plot(M_grid, P_bfc_8s, 'C1-', lw=2, label='BFC m9_fgas-8s')

"""



ax[0].set_xscale('log'); ax[0].set_yscale('linear')
ax[0].set_xlabel(r'$M_{200c}$ [M$_\odot$]')
ax[0].set_ylabel(r'$\int P_e\,dV$ within $r_{200c}$ [eV cm$^{-3}$ Mpc$^3$]')
ax[0].legend(fontsize=8); ax[0].set_title(f'Y-M at z = {z_cat}')


P_bfc_at_cat = np.interp(np.log(m_cent), np.log(M_grid), P_bfc)
P_8s_at_cat  = np.interp(np.log(m_cent), np.log(M_grid), P_bfc_8s)
ax[1].axhline(1, color='k', lw=0.8)
ax[1].axhline(1.12, color='grey', ls=':', lw=1, label='1.12')
ax[1].plot(m_cent, P_bfc_at_cat/P_med, 'C0-o', ms=4, label='BFC m9 / FLAMINGO')
ax[1].plot(m_cent, P_8s_at_cat/P_med,  'C1-o', ms=4, label='BFC fgas-8s / FLAMINGO')
ax[1].set_xscale('log')
ax[1].set_xlabel(r'$M_{200c}$ [M$_\odot$]'); ax[1].set_ylabel('BFC / FLAMINGO')
ax[1].legend(fontsize=8); ax[1].set_title('ratio')
plt.tight_layout()



print(f"\n{'M200c':>10} {'N':>8} {'FLAMINGO':>13} {'BFC m9':>13} {'ratio':>8}")
for i, m in enumerate(m_cent):
    print(f"{m:10.3e} {n_in[i]:8d} {P_med[i]:13.4e} "
          f"{P_bfc_at_cat[i]:13.4e} {P_bfc_at_cat[i]/P_med[i]:8.3f}")

In [ ]:
# zip together the central mass, the number of haloes in that bin, and the ratio between BCF and halo cat.


for m, n, r in zip(m_cent, n_in, P_bfc_at_cat/P_med):
    if m > 1e13: print(f"{m:.2e}  N={n:6d}  ratio={r:.3f}")

In [ ]:

a = 1/(1+z_cat); cos = get_cosmo("m9")
lnM = np.log(10)*hmc._lmass # hmc._lmasss is CCls log10M grid 


# w is the integrand for bPe - we use M^5/3 as a stand in for Y(M).
w = nM(cos, 10**hmc._lmass, a) * bM(cos, 10**hmc._lmass, a) * (10**hmc._lmass)**(5/3)
#print("bPe weighting peaks at M =", 10**hmc._lmass[np.argmax(w)])
#print(f" Distribution of bPe weights: {10**hmc._lmass[w]}")

In [ ]:

# Idea: average the ratio, weighted by each mass's contribution to bPe.

lm = hmc._lmass
Mv = 10**lm # flaw - Mv spans 10^9 - 10^16 (full catalogue) - while m_cent only covers 10^13 - 10^15.
Yf = np.interp(np.log(Mv), np.log(m_cent), P_med) # interp both Y and R to match catalogue values.      
rt = np.interp(np.log(Mv), np.log(m_cent), P_bfc_at_cat/P_med)
w  = nM(cos, Mv, a) * bM(cos, Mv, a) * Yf
print("weighted mean ratio =", np.sum(w*rt)/np.sum(w))

In [ ]:
lm = hmc._lmass # get log mass grid
Mv_all = 10**lm # linearise


sel = (Mv_all > m_cent.min()) & (Mv_all < m_cent.max()) # mask to be in range of bins
Mv = Mv_all[sel] # mask 
Yf = np.interp(np.log(Mv), np.log(m_cent), P_med) # in that range interp at those points        
rt = np.interp(np.log(Mv), np.log(m_cent), P_bfc_at_cat/P_med) # do same for ratio
w  = nM(cos, Mv, a) * bM(cos, Mv, a) * Yf # weight - integrand for bPe - weight determined by how much contributes to bPe.

# print weighted mean ratio in range.
print(f"weighted mean ratio ({m_cent.min():.1e} - {m_cent.max():.1e}) = "
      f"{np.sum(w*rt)/np.sum(w):.4f}")

# fraction of total contribution to bPe in this range.
Y_model = Y_bfc(Mv_all, z_cat, "m9")
w_all   = nM(cos, Mv_all, a) * bM(cos, Mv_all, a) * Y_model

print(f"fraction of total weight in this range: "
      f"{np.sum(w_all[sel])/np.sum(w_all):.4f}")

In [ ]:

m_test = np.array([3e13, 1e14, 3e14, 1e15])
a = 1/(1+z_cat); cos = get_cosmo("m9"); prof = make_mike_flam(z_cat, "m9")
r200 = prof.mass_def.get_radius(cos, m_test, a)/a
conc = cM(cos, m_test, a)
r500 = np.array([float(np.atleast_1d(bfc_f.r500_fct(r,c))[0]) for r,c in zip(r200,conc)])

print(f"{'M200':>10} {'5R500':>10} {'to 1000':>12} {'frac inside':>12}")
for m, xm in zip(m_test, 5*r500/r200):
    inside = float(volint_generic(prof, cos, xm,   m, a, nx=200))
    total  = float(volint_generic(prof, cos, 1000, m, a, nx=200))
    print(f"{m:10.1e} {inside:10.3e} {total:12.3e} {inside/total:12.3f}")